# Projeto 1: Logística - Vendas por Localidade

Análise geográfica para otimização de rotas de entrega

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

## 1.1 Tabela: ft_vendas_consumidor_local

In [0]:
df_pedido_total = spark.table("silver.ft_pedido_total")
df_consumidores = spark.table("silver.ft_consumidores")

df_vendas_local = df_pedido_total.alias("p") \
    .join(df_consumidores.alias("c"), col("p.id_consumidor") == col("c.id_consumidor")) \
    .select(
        col("p.id_pedido"),
        col("p.id_consumidor"),
        col("p.valor_total_pago_brl").cast("decimal(12,2)").alias("valor_total_pedido_brl"),
        col("c.cidade"),
        col("c.estado"),
        col("p.data_pedido").cast("date").alias("data_pedido")
    )

df_vendas_local.write.format("delta").mode("overwrite").saveAsTable("gold.ft_vendas_consumidor_local")

In [0]:
print(f"Registros criados: {spark.table('gold.ft_vendas_consumidor_local').count()}")
spark.table("gold.ft_vendas_consumidor_local").show(5)

## 1.2 View: view_total_compras_por_consumidor

In [0]:
spark.sql("""
    CREATE OR REPLACE VIEW gold.view_total_compras_por_consumidor AS
    SELECT 
        cidade,
        estado,
        COUNT(DISTINCT id_pedido) AS quantidade_vendas,
        CAST(SUM(valor_total_pedido_brl) AS DECIMAL(12,2)) AS valor_total_localidade
    FROM gold.ft_vendas_consumidor_local
    GROUP BY cidade, estado
    ORDER BY valor_total_localidade DESC
""")

In [0]:
spark.sql("SELECT * FROM gold.view_total_compras_por_consumidor LIMIT 26").show()

## 1.3 Consulta: Total de vendas por estado

In [0]:
df_vendas_estado = spark.sql("""
    SELECT 
        estado,
        SUM(quantidade_vendas) AS total_vendas,
        CAST(SUM(valor_total_localidade) AS DECIMAL(12,2)) AS valor_total_estado
    FROM gold.view_total_compras_por_consumidor
    GROUP BY estado
    ORDER BY valor_total_estado DESC
""")

df_vendas_estado.show()

## Validação

In [0]:
print("Objetos criados neste notebook:")
spark.sql("SHOW TABLES IN gold").filter(
    "tableName LIKE '%vendas_consumidor%' OR tableName LIKE '%compras_por_consumidor%'"
).show(truncate=False)